# StatPitch
## Notebook 6 — Unified Data Enrichment

---

Pulls from free data sources and merges them into an enriched dataset for the training pipeline.

| Source | Data | Key needed? |
|---|---|---|
| [football-data.org](https://www.football-data.org/client/register) | International results — Euros, CL | Free key, instant by email |
| [StatsBomb (via statsbombpy)](https://github.com/statsbomb/statsbombpy) | Match & Shot xG — World Cup, Euros, Copa America, AFCON | No key needed (Free Tier) |

*Note: API-Football and Understat have been deprecated in this version in favor of StatsBomb's free event data.*

**Output files saved to your StatPitch Drive folder:**
- `fd_international_matches.csv` — raw international results
- `enriched_matches.csv` — currently mirrors football-data results (ready for future stat merges)
- `xg_data.csv` — StatsBomb match and shot-level xG data
- `team_xg_profiles.csv` — per-team xG averages (merged into `team_stats.json`)

**New features this unlocks in the model:**
`xg_for_avg`, `xg_against_avg`, `xg_overperf`

---
## Setup — Drive, keys, directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

SAVE_DIR    = Path('/content/drive/MyDrive/StatPitch')
PROJECT_DIR = SAVE_DIR / 'statpitch_api'
OUT_DIR     = SAVE_DIR          # all outputs go here

SAVE_DIR.mkdir(exist_ok=True)
(PROJECT_DIR / 'data').mkdir(parents=True, exist_ok=True)

os.chdir(SAVE_DIR)
print(f'Working in:  {SAVE_DIR}')
print(f'Outputs to:  {OUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in:  /content/drive/MyDrive/StatPitch
Outputs to:  /content/drive/MyDrive/StatPitch


In [ ]:
!pip install requests pandas tqdm --quiet

In [ ]:
import re
import json
import time
import math
import random
import shutil
import requests
import pandas as pd
from tqdm import tqdm

print('Libraries loaded!')

Libraries loaded!


### API keys

**Recommended:** Store keys as Colab Secrets (the key icon in the left sidebar) so you never paste them into code.
- Secret name `FOOTBALL_DATA_KEY` → [football-data.org](https://www.football-data.org/client/register)
- Secret name `API_FOOTBALL_KEY` → (Currently optional/bypassed in this pipeline)

If you haven't set up secrets yet, paste the keys directly into the fallback strings below.

In [ ]:
from google.colab import userdata

try:
    FOOTBALL_DATA_KEY = userdata.get('FOOTBALL_DATA_KEY')
    print('football-data.org key loaded from Colab secrets')
except Exception:
    FOOTBALL_DATA_KEY = 'YOUR_FOOTBALL_DATA_ORG_KEY'   # fallback
    print('Paste your football-data.org key into FOOTBALL_DATA_KEY above')

try:
    API_FOOTBALL_KEY = userdata.get('API_FOOTBALL_KEY')
    print('API-Football key loaded from Colab secrets')
except Exception:
    API_FOOTBALL_KEY = 'YOUR_API_FOOTBALL_KEY'          # fallback
    print('Paste your API-Football key into API_FOOTBALL_KEY above')

REQUEST_DELAY = 2.0   # seconds between requests — be polite to free-tier APIs
print(f'\nRequest delay: {REQUEST_DELAY}s')

football-data.org key loaded from Colab secrets
API-Football key loaded from Colab secrets

Request delay: 2.0s


---
## Section 1 — football-data.org

International match results: World Cup, Euros, Champions League.
Free tier covers 12 competitions including WC and EC — exactly what we need.

In [ ]:
FOOTBALL_DATA_BASE    = 'https://api.football-data.org/v4'
FOOTBALL_DATA_HEADERS = {'X-Auth-Token': FOOTBALL_DATA_KEY}

# Competitions on free tier relevant to international football
# Full list: https://docs.football-data.org/general/v4/competition.html
INTL_COMPETITIONS = {
    'EC':  [2024],        # UEFA Euro — current season only on free tier
    'CL':  [2024],        # Champions League — useful as club form proxy
    'WC':  [],            # PAID tier — leave empty, we use martj42 for WC data
}

In [ ]:
def fd_get(endpoint, params=None):
    time.sleep(REQUEST_DELAY)
    r = requests.get(
        f'{FOOTBALL_DATA_BASE}/{endpoint}',
        headers=FOOTBALL_DATA_HEADERS,
        params=params or {},
        timeout=15,
    )
    if r.status_code == 429:
        print('  Rate limited — sleeping 60s')
        time.sleep(60)
        return fd_get(endpoint, params)
    if r.status_code != 200:
        print(f'  WARN: {endpoint} → {r.status_code}: {r.text[:120]}')
        return {}
    return r.json()


def fetch_fd_competition(competition, season):
    data    = fd_get(f'competitions/{competition}/matches', {'season': season})
    matches = data.get('matches', [])
    if not matches:
        return pd.DataFrame()

    rows = []
    for m in matches:
        score = m.get('score', {})
        full  = score.get('fullTime', {})
        rows.append({
            'fd_id':       m.get('id'),
            'date':        m.get('utcDate', '')[:10],
            'competition': m.get('competition', {}).get('name', competition),
            'season':      season,
            'stage':       m.get('stage', ''),
            'status':      m.get('status', ''),
            'home_team':   m.get('homeTeam', {}).get('name', ''),
            'away_team':   m.get('awayTeam', {}).get('name', ''),
            'home_score':  full.get('home'),
            'away_score':  full.get('away'),
            'winner':      score.get('winner'),
        })

    df = pd.DataFrame(rows)
    return df[df['status'] == 'FINISHED'].dropna(subset=['home_score', 'away_score']).copy()

print('football-data.org helpers ready')

football-data.org helpers ready


In [ ]:
fd_frames = []

for comp, seasons in INTL_COMPETITIONS.items():
    for season in seasons:
        print(f'  Fetching {comp} {season}...', end=' ')
        df = fetch_fd_competition(comp, season)
        if not df.empty:
            fd_frames.append(df)
            print(f'{len(df)} matches')
        else:
            print('no data')

fd_matches = pd.concat(fd_frames, ignore_index=True) if fd_frames else pd.DataFrame()
print(f'\nTotal football-data.org matches: {len(fd_matches)}')

if not fd_matches.empty:
    fd_matches.to_csv(OUT_DIR / 'fd_international_matches.csv', index=False)
    print('Saved: fd_international_matches.csv')
    print(fd_matches.head(3).to_string(index=False))

  Fetching EC 2024... 51 matches
  Fetching CL 2024... 189 matches

Total football-data.org matches: 240
Saved: fd_international_matches.csv
 fd_id       date           competition  season       stage   status home_team   away_team  home_score  away_score    winner
428747 2024-06-14 European Championship    2024 GROUP_STAGE FINISHED   Germany    Scotland           5           1 HOME_TEAM
428746 2024-06-15 European Championship    2024 GROUP_STAGE FINISHED   Hungary Switzerland           1           3 AWAY_TEAM
428753 2024-06-15 European Championship    2024 GROUP_STAGE FINISHED     Spain     Croatia           3           0 HOME_TEAM


---
## Section 2 — StatsBomb (statsbombpy)

Replaces Understat to provide high-quality, free Expected Goals (xG) data for international tournaments (World Cups, Euros, Copa America, AFCON).

No API key is needed as we are using the `statsbombpy` package to directly access their free open data tiers.

In [ ]:
!pip install statsbombpy --quiet

from statsbombpy import sb
import pandas as pd

# See all free competitions available
from statsbombpy import sb
comps = sb.competitions()

# Filter to competitions relevant to international football
keywords = ['World Cup','Euro','Copa','Nations','Africa','CONCACAF','International']
intl = comps[comps['competition_name'].str.contains('|'.join(keywords), case=False)]
print(intl[['competition_id','season_id','competition_name','season_name']].to_string(index=False))

 competition_id  season_id       competition_name season_name
           1267        107 African Cup of Nations        2023
            223        282           Copa America        2024
             87         84           Copa del Rey   1983/1984
             87        268           Copa del Rey   1982/1983
             87        279           Copa del Rey   1977/1978
           1470        274     FIFA U20 World Cup        1979
             43        106         FIFA World Cup        2022
             43          3         FIFA World Cup        2018
             43         55         FIFA World Cup        1990
             43         54         FIFA World Cup        1986
             43         51         FIFA World Cup        1974
             43        272         FIFA World Cup        1970
             43        270         FIFA World Cup        1962
             43        269         FIFA World Cup        1958
             55        282              UEFA Euro        2024
        

In [ ]:
STATSBOMB_COMPS = [
    (43,   106, 'FIFA World Cup 2022'),
    (43,   3,   'FIFA World Cup 2018'),
    (43,   55,  'FIFA World Cup 1990'),
    (43,   54,  'FIFA World Cup 1986'),
    (55,   282, 'UEFA Euro 2024'),
    (55,   43,  'UEFA Euro 2020'),
    (223,  282, 'Copa America 2024'),
    (1267, 107, 'African Cup of Nations 2023'),
]

us_frames = []

for comp_id, season_id, name in STATSBOMB_COMPS:
    print(f'  Fetching {name}...', end=' ')
    try:
        matches = sb.matches(competition_id=comp_id, season_id=season_id)
        if matches.empty:
            print('no data')
            continue
        rows = []
        for _, m in matches.iterrows():
            rows.append({
                'date':       str(m['match_date'])[:10],
                'home_team':  m['home_team'],
                'away_team':  m['away_team'],
                'home_goals': int(m['home_score']),
                'away_goals': int(m['away_score']),
                'home_xg':    float(m.get('home_xg') or 0),
                'away_xg':    float(m.get('away_xg') or 0),
                'league':     name,
                'season':     season_id,
                'comp_id':    comp_id,
            })
        df = pd.DataFrame(rows)
        us_frames.append(df)
        print(f'{len(df)} matches')
    except Exception as e:
        print(f'error: {e}')

xg_data = pd.concat(us_frames, ignore_index=True) if us_frames else pd.DataFrame()
print(f'\nTotal StatsBomb rows: {len(xg_data)}')
print(xg_data.groupby('league')[['home_xg','away_xg']].mean().round(3))

  Fetching FIFA World Cup 2022... 64 matches
  Fetching FIFA World Cup 2018... 64 matches
  Fetching FIFA World Cup 1990... 1 matches
  Fetching FIFA World Cup 1986... 3 matches
  Fetching UEFA Euro 2024... 51 matches
  Fetching UEFA Euro 2020... 51 matches
  Fetching Copa America 2024... 32 matches
  Fetching African Cup of Nations 2023... 52 matches

Total StatsBomb rows: 318
                             home_xg  away_xg
league                                       
African Cup of Nations 2023      0.0      0.0
Copa America 2024                0.0      0.0
FIFA World Cup 1986              0.0      0.0
FIFA World Cup 1990              0.0      0.0
FIFA World Cup 2018              0.0      0.0
FIFA World Cup 2022              0.0      0.0
UEFA Euro 2020                   0.0      0.0
UEFA Euro 2024                   0.0      0.0


In [ ]:
# WC 2018 and 2022 have full event data — pull exact shot xG per match
import warnings
warnings.filterwarnings('ignore', message='.*credentials were not supplied.*')

ENRICH_COMPS = [
    (43, 106, 'FIFA World Cup 2022'),
    (43, 3,   'FIFA World Cup 2018'),
]

enriched_xg = []

for comp_id, season_id, name in ENRICH_COMPS:
    matches = sb.matches(competition_id=comp_id, season_id=season_id)
    print(f'Shot-level xG for {len(matches)} matches — {name}...')

    for _, m in matches.iterrows():
        mid = m['match_id']
        try:
            events = sb.events(match_id=mid)
            shots  = events[events['type'] == 'Shot']
            if shots.empty:
                continue
            home    = m['home_team']
            away    = m['away_team']
            home_xg = shots[shots['team'] == home]['shot_statsbomb_xg'].sum()
            away_xg = shots[shots['team'] == away]['shot_statsbomb_xg'].sum()
            enriched_xg.append({
                'date':       str(m['match_date'])[:10],
                'home_team':  home,
                'away_team':  away,
                'home_goals': int(m['home_score']),
                'away_goals': int(m['away_score']),
                'home_xg':    round(float(home_xg), 3),
                'away_xg':    round(float(away_xg), 3),
                'league':     name,
                'season':     season_id,
            })
        except Exception:
            continue

    print(f'  Done — {sum(1 for r in enriched_xg if r["season"]==season_id)} matches enriched')

if enriched_xg:
    df_shot_xg = pd.DataFrame(enriched_xg)
    # Replace the match-level WC rows with the more accurate shot-level ones
    xg_data = xg_data[~xg_data['league'].str.contains('World Cup 2022|World Cup 2018')]
    xg_data = pd.concat([xg_data, df_shot_xg], ignore_index=True).reset_index(drop=True)
    print(f'\nFinal total xG rows: {len(xg_data)}')

# Remount Drive (will be instant if still connected, or ask to reconnect)
drive.mount('/content/drive', force_remount=True)

SAVE_DIR = Path('/content/drive/MyDrive/StatPitch')
OUT_DIR  = SAVE_DIR
SAVE_DIR.mkdir(exist_ok=True)
os.chdir(SAVE_DIR)

# Save the xg_data that's already in memory
xg_data.to_csv(OUT_DIR / 'xg_data.csv', index=False)
print(f'Saved {len(xg_data)} rows to xg_data.csv')
print(xg_data.groupby('league')[['home_xg','away_xg']].mean().round(3))

Shot-level xG for 64 matches — FIFA World Cup 2022...
  Done — 64 matches enriched
Shot-level xG for 64 matches — FIFA World Cup 2018...
  Done — 64 matches enriched

Final total xG rows: 318
Mounted at /content/drive
Saved 318 rows to xg_data.csv
                             home_xg  away_xg
league                                       
African Cup of Nations 2023    0.000    0.000
Copa America 2024              0.000    0.000
FIFA World Cup 1986            0.000    0.000
FIFA World Cup 1990            0.000    0.000
FIFA World Cup 2018            1.760    1.417
FIFA World Cup 2022            1.506    1.432
UEFA Euro 2020                 0.000    0.000
UEFA Euro 2024                 0.000    0.000


---
## Section 3 — Team xG Profiles

Aggregate match and shot-level xG into per-team averages.

`xg_overperf` = a team scores more goals than their Expected Goals (xG) predicts. This provides a measurable proxy for clinical finishing quality and goalkeeper performance across national teams.

In [ ]:
if not xg_data.empty:
    home_view = xg_data.rename(columns={
        'home_team': 'team', 'home_xg': 'xg_for',
        'away_xg': 'xg_against', 'home_goals': 'goals_for',
        'away_goals': 'goals_against'
    })[['date','team','xg_for','xg_against','goals_for','goals_against','league','season']]

    away_view = xg_data.rename(columns={
        'away_team': 'team', 'away_xg': 'xg_for',
        'home_xg': 'xg_against', 'away_goals': 'goals_for',
        'home_goals': 'goals_against'
    })[['date','team','xg_for','xg_against','goals_for','goals_against','league','season']]

    team_matches = pd.concat([home_view, away_view], ignore_index=True)

    team_xg_profiles = (
        team_matches.groupby(['team','season'])
        .agg(
            matches           = ('xg_for',       'count'),
            avg_xg_for        = ('xg_for',       'mean'),
            avg_xg_against    = ('xg_against',   'mean'),
            avg_goals_for     = ('goals_for',     'mean'),
            avg_goals_against = ('goals_against', 'mean'),
        )
        .reset_index()
    )

    # xg_overperf: scores more than xG suggests → clinical finishing
    team_xg_profiles['xg_overperf'] = (
        team_xg_profiles['avg_goals_for'] - team_xg_profiles['avg_xg_for']
    )

    # Keep most recent season per team for joining into team_stats
    latest_xg = (
        team_xg_profiles
        .sort_values('season', ascending=False)
        .groupby('team')
        .first()
        .reset_index()
    )

    latest_xg.to_csv(OUT_DIR / 'team_xg_profiles.csv', index=False)
    print(f'Profiles for {len(latest_xg)} clubs saved: team_xg_profiles.csv')
    print()
    print('Top 10 by avg xG (most recent season):')
    print(
        latest_xg.nlargest(10, 'avg_xg_for')
        [['team','season','matches','avg_xg_for','avg_xg_against','xg_overperf']]
        .to_string(index=False)
    )
else:
    print('No xG data — skipping profile build')
    latest_xg = pd.DataFrame()

Profiles for 76 clubs saved: team_xg_profiles.csv

Top 10 by avg xG (most recent season):
        team  season  matches  avg_xg_for  avg_xg_against  xg_overperf
       Japan     106        4    1.846000        1.953250    -0.596000
     Iceland       3        3    1.572333        1.355667    -0.905667
        Iran     106        3    1.277000        1.257667     0.056333
Saudi Arabia     106        3    1.067000        2.182667    -0.067000
 South Korea     106        4    0.901250        1.458250     0.348750
       Wales     106        3    0.739667        1.657000    -0.406333
       Qatar     106        3    0.469333        1.257000    -0.136000
   Australia     106        4    0.394500        1.610250     0.605500
     Albania     282        3    0.000000        0.000000     1.000000
     Algeria     107        3    0.000000        0.000000     1.000000


---
## Section 4 — Merge into enriched_matches.csv

Historically joined football-data.org results with API-Football stats. Currently, API-Football data fetching is bypassed in the code (`af_stats` is set to empty).

This section safely passes through the `fd_matches` data to generate our base `enriched_matches.csv` file, ensuring downstream pipeline compatibility while preventing missing data errors.

In [ ]:
def norm(s):
    return str(s).lower().strip() if s else ''

af_stats = pd.DataFrame()
print('af_stats set to empty — merge will use fd_matches only')

if not fd_matches.empty and not af_stats.empty:
    fd_matches['_home_key'] = fd_matches['home_team'].apply(norm)
    fd_matches['_away_key'] = fd_matches['away_team'].apply(norm)
    af_stats['_home_key']   = af_stats['home_team'].apply(norm)
    af_stats['_away_key']   = af_stats['away_team'].apply(norm)

    enriched = fd_matches.merge(
        af_stats.drop(columns=['home_team','away_team','home_score',
                                'away_score','competition'], errors='ignore'),
        on=['date','_home_key','_away_key'],
        how='left',
    ).drop(columns=['_home_key','_away_key'])

    match_rate = enriched['home_shots'].notna().mean() * 100
    print(f'fd_matches rows : {len(fd_matches)}')
    print(f'af_stats rows   : {len(af_stats)}')
    print(f'Merged rows     : {len(enriched)}')
    print(f'Stat coverage   : {match_rate:.1f}% of matches have shot data')

elif not fd_matches.empty:
    enriched = fd_matches.copy()
    print('API-Football stats unavailable — using fd_matches only')
else:
    enriched = pd.DataFrame()
    print('WARNING: No data fetched. Check your API keys above.')

if not enriched.empty:
    enriched.to_csv(OUT_DIR / 'enriched_matches.csv', index=False)
    print('\nSaved: enriched_matches.csv')
    print(enriched.head(3).to_string(index=False))

af_stats set to empty — merge will use fd_matches only
API-Football stats unavailable — using fd_matches only

Saved: enriched_matches.csv
 fd_id       date           competition  season       stage   status home_team   away_team  home_score  away_score    winner
428747 2024-06-14 European Championship    2024 GROUP_STAGE FINISHED   Germany    Scotland           5           1 HOME_TEAM
428746 2024-06-15 European Championship    2024 GROUP_STAGE FINISHED   Hungary Switzerland           1           3 AWAY_TEAM
428753 2024-06-15 European Championship    2024 GROUP_STAGE FINISHED     Spain     Croatia           3           0 HOME_TEAM


---
## Section 5 — NaN Safety Guard

Prevent NaN values from leaking into downstream notebooks and silently corrupting features.

In [ ]:
def check_nan_leak(df, name):
    if df is None or df.empty:
        print(f'  {name}: EMPTY (skipped)')
        return
    nan_cols = df.columns[df.isna().any()].tolist()
    if nan_cols:
        for col in nan_cols:
            pct = df[col].isna().mean() * 100
            print(f'  WARNING  {name}.{col}: {pct:.1f}% NaN')
    else:
        print(f'  OK  {name}: no NaN columns')

print('NaN audit:')
check_nan_leak(enriched,  'enriched_matches')
check_nan_leak(xg_data,   'xg_data')
check_nan_leak(latest_xg, 'team_xg_profiles')

# Fill stat NaNs with 0 (safe default for shots/corners/possession)
if not enriched.empty:
    stat_cols = [c for c in enriched.columns if any(
        k in c for k in ['shots','possession','corners','fouls',
                          'yellows','reds','saves','passes']
    )]
    enriched[stat_cols] = enriched[stat_cols].fillna(0)
    enriched.to_csv(OUT_DIR / 'enriched_matches.csv', index=False)
    print('\nStat NaNs filled with 0 and re-saved')

NaN audit:
  OK  enriched_matches: no NaN columns
  WARNING  xg_data.comp_id: 40.3% NaN
  OK  team_xg_profiles: no NaN columns

Stat NaNs filled with 0 and re-saved


---
## Section 6 — Integrate with StatPitch pipeline

Merge the new xG profiles into `team_stats.json` so the API and training pipeline can use them automatically. This updates the same file the predictor loads on startup.

In [ ]:
team_stats_path = OUT_DIR / 'team_stats.json'

if not team_stats_path.exists():
    print('team_stats.json not found — run notebooks 01-04 first, then re-run this cell')
else:
    with open(team_stats_path) as f:
        team_stats = json.load(f)

    updated = 0
    skipped = 0

    if not latest_xg.empty:
        xg_lookup = latest_xg.set_index('team').to_dict('index')

        for team, stats in team_stats.items():
            if team in xg_lookup:
                xg = xg_lookup[team]
                stats['club_xg_for']     = round(float(xg.get('avg_xg_for',   0)), 3)
                stats['club_xg_against'] = round(float(xg.get('avg_xg_against',0)), 3)
                stats['club_xg_overperf']= round(float(xg.get('xg_overperf',  0)), 3)
                updated += 1
            else:
                stats['club_xg_for']     = stats.get('gs_avg5', 1.3)
                stats['club_xg_against'] = stats.get('gc_avg5', 1.3)
                stats['club_xg_overperf']= 0.0
                skipped += 1

    with open(team_stats_path, 'w') as f:
        json.dump(team_stats, f, indent=2)

    # Copy to API data folder
    shutil.copy2(team_stats_path, PROJECT_DIR / 'data' / 'team_stats.json')

    print(f'team_stats.json updated:')
    print(f'  Teams with club xG data:    {updated}')
    print(f'  Teams using goal proxy:     {skipped}')
    print(f'  Copied to statpitch_api/data/')

team_stats.json updated:
  Teams with club xG data:    73
  Teams using goal proxy:     210
  Copied to statpitch_api/data/


In [ ]:
# Copy output files to the API data folder so they're available for training
files_to_copy = [
    'enriched_matches.csv',
    'af_match_stats.csv',
    'xg_data.csv',
    'team_xg_profiles.csv',
]

api_data = PROJECT_DIR / 'data'
for fname in files_to_copy:
    src = OUT_DIR / fname
    if src.exists():
        shutil.copy2(src, api_data / fname)
        print(f'Copied: {fname}')
    else:
        print(f'Skipped (not produced): {fname}')

Copied: enriched_matches.csv
Skipped (not produced): af_match_stats.csv
Copied: xg_data.csv
Copied: team_xg_profiles.csv


---
## Summary

In [ ]:
enriched_len = len(enriched) if not enriched.empty else 0
xg_len       = len(xg_data)  if not xg_data.empty  else 0
latest_len   = len(latest_xg) if not latest_xg.empty else 0

print('=' * 60)
print('NOTEBOOK 06 COMPLETE')
print('=' * 60)
print()
print(f'Output files in {OUT_DIR}:')
print(f'  enriched_matches.csv       {enriched_len} international matches + stats')
print(f'  fd_international_matches.csv  raw football-data.org results')
print(f'  af_match_stats.csv         raw API-Football shot/possession stats')
print(f'  xg_data.csv                Understat club xG ({xg_len} matches)')
print(f'  team_xg_profiles.csv       per-team xG averages ({latest_len} teams)')
print()
print('API usage:')
print(f'  football-data.org : 10 req/min (auto-throttled)')
print(f'  Understat         : no key required')
print()
print('New features available in the next training run:')
print('  avg_shots_last5      rolling shot volume (af_match_stats)')
print('  avg_possession_last5 rolling possession  (af_match_stats)')
print('  xg_for_avg           avg xG generated   (team_xg_profiles)')
print('  xg_against_avg       avg xG conceded    (team_xg_profiles)')
print('  xg_overperf          finishing quality  (team_xg_profiles)')
print()
print('Next: re-run Notebook 05 to retrain models with these new features')

NOTEBOOK 06 COMPLETE

Output files in /content/drive/MyDrive/StatPitch:
  enriched_matches.csv       240 international matches + stats
  fd_international_matches.csv  raw football-data.org results
  af_match_stats.csv         raw API-Football shot/possession stats
  xg_data.csv                Understat club xG (318 matches)
  team_xg_profiles.csv       per-team xG averages (76 teams)

API usage:
  football-data.org : 10 req/min (auto-throttled)
  Understat         : no key required

New features available in the next training run:
  avg_shots_last5      rolling shot volume (af_match_stats)
  avg_possession_last5 rolling possession  (af_match_stats)
  xg_for_avg           avg xG generated   (team_xg_profiles)
  xg_against_avg       avg xG conceded    (team_xg_profiles)
  xg_overperf          finishing quality  (team_xg_profiles)

Next: re-run Notebook 05 to retrain models with these new features
